# Data Wrangling & EDA — Part 2
## CPI Sektoral, Mortalitas, Gaji, Investasi

In [ ]:
import re, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import linregress
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize':(14,5),'font.size':11,'axes.grid':True,'grid.alpha':0.3})
ROOT = Path('.').resolve()
if ROOT.name in ('scripts','notebooks'): ROOT = ROOT.parent
RAW, PROC = ROOT/'data'/'raw', ROOT/'data'/'processed'

def parse_num(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s in ('-','','nan','#N/A'): return np.nan
    s = re.sub(r'[^\d.,-]','',s)
    if not s: return np.nan
    has_c, has_d = ',' in s, '.' in s
    if has_c and has_d:
        if s.rfind(',')>s.rfind('.'): s=s.replace('.','').replace(',','.')
        else: s=s.replace(',','')
    elif has_c: s=s.replace(',','.')
    elif has_d:
        parts=s.split('.')
        if len(parts)>1 and all(len(p)==3 for p in parts[1:]): s=s.replace('.','')
    try: return float(s)
    except: return np.nan

def parse_dot(val):
    if pd.isna(val): return np.nan
    s = re.sub(r'[^\d.-]','',str(val).strip())
    try: return float(s)
    except: return np.nan

def find_indonesia_row(fp):
    df=pd.read_csv(fp,header=0,dtype=str,encoding='utf-8-sig')
    col=df.columns[0]; mask=df[col].str.strip().str.upper()=='INDONESIA'
    if not mask.any(): mask=df[col].str.strip().str.upper().str.contains('INDONESIA',na=False)
    return df[mask].iloc[0]

def monthly_from_bps(series):
    recs=[]
    for col,val in series.items():
        col=str(col).strip()
        if re.match(r'^\d{2}/\d{4}$',col):
            mm,yyyy=col.split('/'); recs.append({'year_month':f'{yyyy}-{mm}','index_value':parse_num(val)})
    if not recs: return pd.DataFrame(columns=['year_month','index_value'])
    df=pd.DataFrame(recs); df['year_month']=pd.to_datetime(df['year_month'],format='%Y-%m')
    return df.sort_values('year_month').reset_index(drop=True)

def impute(s): return s.interpolate('linear').ffill().bfill()
# Load CPI umum for multiplier
cpi = pd.read_csv(PROC/'cpi_monthly.csv')
print('Helpers loaded. CPI umum loaded.')

---
## 2. CPI Sektoral (Makanan, Kesehatan, Pendidikan)

### Tujuan
Menghitung **multiplier inflasi sektoral** — berapa kali lipat inflasi sektor tertentu dibanding inflasi umum.

### Penanganan Data Kosong
Sektor Makanan memiliki 2 bulan data kosong (tanda `-`). Diisi dengan interpolasi linear.

In [ ]:
SEKTOR = {
    'makanan': RAW/'CPI Sektor'/'Sektor 01 Makanan',
    'kesehatan': RAW/'CPI Sektor'/'Sektor 05 Kesehatan',
    'pendidikan': RAW/'CPI Sektor'/'Sektor 09 Pendidikan',
}
sektor_data = {}
for name, folder in SEKTOR.items():
    pieces = []
    for f in sorted(folder.glob('*.csv')):
        try:
            row = find_indonesia_row(f)
            df = monthly_from_bps(row)
            if not df.empty: pieces.append(df)
        except: pass
    if not pieces: continue
    df = pd.concat(pieces).drop_duplicates('year_month').sort_values('year_month').reset_index(drop=True)
    n_miss = df.index_value.isna().sum()
    if n_miss:
        print(f"  {name}: {n_miss} data kosong ditemukan -> diimputasi")
        df['index_value'] = impute(df['index_value'])
    df['mom'] = impute(df.index_value.pct_change(1)*100)
    df['yoy'] = impute(df.index_value.pct_change(12)*100)
    sektor_data[name] = df
    print(f"  {name}: {len(df)} bulan OK")

### Visualisasi: Inflasi YoY per Sektor vs Umum

In [ ]:
fig, ax = plt.subplots(figsize=(14,6))
colors = {'makanan':'#E53935','kesehatan':'#1E88E5','pendidikan':'#43A047'}
for name, df in sektor_data.items():
    ax.plot(df.year_month, df.yoy, '-', color=colors[name], lw=1.5, label=f'Sektor {name.title()}', alpha=0.8)
# CPI umum
cpi_m = pd.read_csv(PROC/'cpi_monthly.csv')
cpi_m['year_month'] = pd.to_datetime(cpi_m['year_month'])
ax.plot(cpi_m.year_month, cpi_m.inflation_yoy_pct, 'k--', lw=2, label='CPI Umum', alpha=0.7)
ax.axhline(0, color='gray', ls='-', alpha=0.3)
ax.set_title('Perbandingan Inflasi YoY: Sektoral vs Umum', fontsize=14, fontweight='bold')
ax.set_ylabel('Inflasi YoY (%)')
ax.legend()
plt.tight_layout(); plt.show()

### Identifikasi Outlier & Perhitungan Multiplier

**Bulan anomali yang dibuang dari perhitungan multiplier:**
- Mar-Jun 2020: Pandemi COVID, distorsi supply-demand
- Jan-Feb 2021: PPKM ketat
- Sep-Nov 2022: Kenaikan BBM

**Alasan pakai Median (bukan Mean):**
Mean rusak oleh 1 outlier. Median robust terhadap pencilan.

In [ ]:
COVID_MONTHS = {'2020-03','2020-04','2020-05','2020-06','2021-01','2021-02','2022-09','2022-10','2022-11'}
cpi_ref = cpi_m[['year_month','inflation_yoy_pct']].rename(columns={'inflation_yoy_pct':'umum_yoy'})
cpi_ref['ym'] = cpi_ref.year_month.dt.strftime('%Y-%m')

mult_results = []
fig, axes = plt.subplots(1, 3, figsize=(18,5))
for i, (name, df) in enumerate(sektor_data.items()):
    df['ym'] = df.year_month.dt.strftime('%Y-%m')
    m = df[['ym','yoy']].merge(cpi_ref[['ym','umum_yoy']], on='ym', how='inner')
    m = m.dropna(subset=['yoy','umum_yoy'])
    
    # Hitung rasio SEMUA bulan
    m['ratio'] = m.yoy / m.umum_yoy
    m.loc[m.ratio.abs()>10, 'ratio'] = np.nan  # clip extreme
    
    # Tandai outlier
    m['is_outlier'] = m.ym.isin(COVID_MONTHS)
    clean = m[~m.is_outlier & (m.umum_yoy.abs()>0.1)].dropna(subset=['ratio'])
    
    median_mult = clean.ratio.median()
    mean_mult = clean.ratio.mean()
    mult_results.append({'sektor':name, 'multiplier_median':round(median_mult,3), 'multiplier_mean':round(mean_mult,3), 'n_obs':len(clean)})
    
    # Plot
    ax = axes[i]
    ax.hist(clean.ratio, bins=20, alpha=0.7, color=colors[name], edgecolor='white')
    ax.axvline(median_mult, color='black', ls='-', lw=2, label=f'Median={median_mult:.3f}')
    ax.axvline(mean_mult, color='orange', ls='--', lw=2, label=f'Mean={mean_mult:.3f}')
    ax.axvline(1.0, color='gray', ls=':', alpha=0.5)
    ax.set_title(f'{name.title()} — Distribusi Rasio', fontweight='bold')
    ax.set_xlabel('Rasio (Sektor YoY / Umum YoY)')
    ax.legend(fontsize=9)

plt.suptitle('Distribusi Multiplier Inflasi Sektoral (Outlier COVID dibuang)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

df_mult = pd.DataFrame(mult_results)
print("\nHasil Multiplier:")
print(df_mult.to_string(index=False))
print("\nMedian lebih robust karena tidak terpengaruh outlier yang lolos filter.")
df_mult.to_csv(PROC/'cpi_sektor_multiplier.csv', index=False)

In [ ]:
all_sektor = []
for name, df in sektor_data.items():
    df['sektor'] = name
    df['year_month'] = df.year_month.dt.strftime('%Y-%m')
    all_sektor.append(df[['sektor','year_month','index_value','mom','yoy']].rename(columns={'mom':'inflation_mom_pct','yoy':'inflation_yoy_pct'}))
pd.concat(all_sektor).to_csv(PROC/'cpi_sektor_monthly.csv', index=False)
print("Saved: cpi_sektor_monthly.csv, cpi_sektor_multiplier.csv")

---
## 3. Tabel Mortalitas & A/E Ratio (TMPI 2023)

### Sumber
TMPI (Tabel Mortalitas Penduduk Indonesia) 2023, diterbitkan oleh BPJS/BPS.

### Apa itu A/E Ratio?
**Actual / Expected** — berapa kali lipat kematian aktual vs prediksi tabel standar.
- A/E > 1: kematian aktual lebih tinggi dari tabel standar
- A/E < 1: kematian aktual lebih rendah

In [ ]:
def pct_to_float(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s.endswith('%'):
        try: return float(s[:-1])/100
        except: return np.nan
    return parse_num(val)

MORT_DIR = RAW/'Tabel Mortalitas Indonesia (TMPI JKN 2023)'
mort_file = MORT_DIR/'Tabel_Mortalitas_Penduduk_Indonesia_2023.csv'
df_mort = pd.read_csv(mort_file, dtype=str, encoding='utf-8-sig')
df_mort = df_mort.dropna(subset=['age'])
df_mort = df_mort[df_mort.age.str.strip()!='']

NUM_COLS = ['age','exposure_male','dx_male','expectedlife_male','qx_male','px_male',
            'exposure_female','dx_female','expectedlife_female','qx_female','px_female']
for c in NUM_COLS:
    if c in df_mort.columns: df_mort[c] = df_mort[c].apply(parse_num)

AE_COLS = [c for c in df_mort.columns if 'A/E' in c]
for c in AE_COLS: df_mort[c] = df_mort[c].apply(pct_to_float)

df_mort['age'] = df_mort.age.apply(parse_num).astype(int)
df_mort = df_mort.sort_values('age').reset_index(drop=True)

ae_m = [c for c in AE_COLS if 'LK' in c.upper()]
ae_f = [c for c in AE_COLS if 'PR' in c.upper()]
df_mort['ae_avg_male'] = df_mort[ae_m].mean(axis=1).round(4)
df_mort['ae_avg_female'] = df_mort[ae_f].mean(axis=1).round(4)

print(f"Mortalitas: {len(df_mort)} usia (0-{df_mort.age.max()})")
print(f"A/E Ratio kolom: {AE_COLS}")
print(f"Contoh A/E usia 0: Male={df_mort.loc[0,'ae_avg_male']}, Female={df_mort.loc[0,'ae_avg_female']}")

### Visualisasi: Probabilitas Kematian (qx) dan A/E Ratio

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# qx curve
ax1 = axes[0]
ax1.semilogy(df_mort.age, df_mort.qx_male, 'b-', lw=1.5, label='Laki-laki (qx)')
ax1.semilogy(df_mort.age, df_mort.qx_female, 'r-', lw=1.5, label='Perempuan (qx)')
ax1.set_title('Probabilitas Kematian per Usia (qx) — Log Scale', fontweight='bold')
ax1.set_xlabel('Usia'); ax1.set_ylabel('qx (log scale)')
ax1.legend(); ax1.set_xlim(0,111)

# A/E ratio
ax2 = axes[1]
ax2.plot(df_mort.age, df_mort.ae_avg_male, 'b-', lw=1.5, alpha=0.7, label='A/E Laki-laki (avg)')
ax2.plot(df_mort.age, df_mort.ae_avg_female, 'r-', lw=1.5, alpha=0.7, label='A/E Perempuan (avg)')
ax2.axhline(1.0, color='gray', ls='--', alpha=0.7, label='A/E = 1 (expected)')
ax2.set_title('A/E Ratio Rata-rata 2018-2022', fontweight='bold')
ax2.set_xlabel('Usia'); ax2.set_ylabel('A/E Ratio')
ax2.legend(); ax2.set_xlim(0,111); ax2.set_ylim(0.5, 3.0)

plt.tight_layout(); plt.show()
print("A/E > 1 di hampir semua usia: mortalitas aktual Indonesia LEBIH TINGGI dari tabel standar.")

In [ ]:
mort_keep = [c for c in NUM_COLS if c in df_mort.columns]
df_mort[mort_keep].to_csv(PROC/'mortality_clean.csv', index=False)
ae_keep = ['age'] + AE_COLS + ['ae_avg_male','ae_avg_female']
df_mort[[c for c in ae_keep if c in df_mort.columns]].to_csv(PROC/'ae_ratio_clean.csv', index=False)
print("Saved: mortality_clean.csv, ae_ratio_clean.csv")

---
## 4. Rata-rata Gaji per Sektor (BPS)

### Metode Perhitungan Growth:
1. **CAGR** (seluruh tahun termasuk COVID) -> `growth_with_covid`
2. **Log-Linear Regression** (exclude 2020-2021) -> `growth_normal`

In [ ]:
SAL_DIR = RAW/'Rata-rata Upah Gaji per Sektor 2015-2026 (February Data - BPS Rakernas)'
sal_file = SAL_DIR/'Rata-rata upah gaji per sektor 2015 - 2026 BPS.csv'
df_sal = pd.read_csv(sal_file, dtype=str, encoding='utf-8-sig')
sektor_col = df_sal.columns[0]
year_cols = [c for c in df_sal.columns if re.match(r'^\d{4}$', str(c).strip())]
for c in year_cols: df_sal[c] = df_sal[c].apply(parse_num)
df_sal.to_csv(PROC/'salary_clean.csv', index=False)
print(f"Gaji: {len(df_sal)} sektor, tahun {year_cols[0]}-{year_cols[-1]}")
df_sal.head()

In [ ]:
COVID_YEARS = {2020, 2021}
growth_rows = []
for _, row in df_sal.iterrows():
    sektor = str(row[sektor_col]).strip()
    vals = {int(y): row[y] for y in year_cols if not pd.isna(row[y]) and row[y]>0}
    if len(vals)<3: continue
    ys = sorted(vals.keys())
    ymin, ymax = ys[0], ys[-1]
    n = ymax - ymin
    cagr = (vals[ymax]/vals[ymin])**(1/n)-1 if n>0 else np.nan
    clean = [(y,vals[y]) for y in ys if y not in COVID_YEARS]
    if len(clean)>=3:
        xs = np.array([y-clean[0][0] for y,_ in clean])
        lny = np.log([v for _,v in clean])
        slope,*_ = linregress(xs, lny)
        trend = np.exp(slope)-1
    else: trend = cagr
    growth_rows.append({'sektor':sektor,'growth_normal':round(trend,4),'growth_with_covid':round(cagr,4),'year_start':ymin,'year_end':ymax})

df_growth = pd.DataFrame(growth_rows)
df_growth.to_csv(PROC/'salary_growth.csv', index=False)
print(df_growth[['sektor','growth_normal','growth_with_covid']].to_string(index=False))

### Visualisasi: Tren Gaji per Sektor & Efek COVID

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap-style: growth normal
ax1 = axes[0]
sorted_g = df_growth.sort_values('growth_normal', ascending=True)
colors_g = ['#E53935' if g<0.02 else '#FF9800' if g<0.04 else '#4CAF50' for g in sorted_g.growth_normal]
bars = ax1.barh(range(len(sorted_g)), sorted_g.growth_normal*100, color=colors_g, edgecolor='white')
ax1.set_yticks(range(len(sorted_g)))
ax1.set_yticklabels([s[:35] for s in sorted_g.sektor], fontsize=8)
ax1.set_xlabel('Growth Rate (%)')
ax1.set_title('Growth Gaji Normal (excl. COVID)', fontweight='bold')
ax1.xaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f%%'))

# Perbandingan normal vs covid
ax2 = axes[1]
x = range(len(df_growth))
w = 0.35
ax2.barh([i-w/2 for i in x], df_growth.growth_normal*100, w, label='Normal (excl COVID)', color='#4CAF50', alpha=0.8)
ax2.barh([i+w/2 for i in x], df_growth.growth_with_covid*100, w, label='With COVID', color='#FF5722', alpha=0.8)
ax2.set_yticks(x)
ax2.set_yticklabels([s[:30] for s in df_growth.sektor], fontsize=7)
ax2.set_xlabel('Growth Rate (%)')
ax2.set_title('Normal vs COVID Growth', fontweight='bold')
ax2.legend(fontsize=9)
ax2.xaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f%%'))

plt.tight_layout(); plt.show()

---
## 5. Data Investasi (IHSG, Obligasi 3Y & 10Y)

In [ ]:
INV = RAW/'Investasi'
# IHSG
ihsg_f = list((INV/'Monthly IHSG').glob('*.csv'))[0]
df_ihsg = pd.read_csv(ihsg_f, dtype=str)
df_ihsg['Date'] = pd.to_datetime(df_ihsg['Date'], format='mixed', dayfirst=False)
df_ihsg['Close'] = df_ihsg['Close'].apply(parse_dot)
df_ihsg = df_ihsg[['Date','Close']].dropna().sort_values('Date').reset_index(drop=True)
df_ihsg = df_ihsg[df_ihsg.Date.dt.year<=2025]
df_ihsg['mom'] = df_ihsg.Close.pct_change(1)*100
df_ihsg['yoy'] = df_ihsg.Close.pct_change(12)*100
df_ihsg['year_month'] = df_ihsg.Date.dt.strftime('%Y-%m')
df_ihsg[['year_month','Close','mom','yoy']].rename(columns={'mom':'return_mom_pct','yoy':'return_yoy_pct'}).to_csv(PROC/'ihsg_monthly.csv', index=False)

ihsg_ann = df_ihsg[df_ihsg.Date.dt.month==12].copy()
ihsg_ann['year'] = ihsg_ann.Date.dt.year
ihsg_ann[['year','yoy']].rename(columns={'yoy':'ihsg_annual_return_pct'}).to_csv(PROC/'ihsg_annual.csv', index=False)

print(f"IHSG: {len(df_ihsg)} bulan, Mean YoY={df_ihsg.yoy.dropna().mean():.1f}%, Std={df_ihsg.yoy.dropna().std():.1f}%")

# Obligasi
def load_yield(folder, label):
    fld = INV/folder
    files = list(fld.glob('*.csv')) if fld.exists() else []
    if not files: return None
    df = pd.read_csv(files[0], dtype=str)
    df['Date'] = pd.to_datetime(df['Date'], format='mixed')
    df['yield_pct'] = df['Close'].apply(parse_dot)
    df = df[['Date','yield_pct']].dropna().sort_values('Date')
    df = df[df.Date.dt.year<=2025]
    df['year_month'] = df.Date.dt.strftime('%Y-%m')
    print(f"{label}: {len(df)} bulan, avg={df.yield_pct.mean():.2f}%")
    return df[['year_month','yield_pct']]

ob10 = load_yield('10 Tahun Obligasi 2015 - 2026','Obligasi 10Y')
ob3 = load_yield('3 Tahun Obligasi 2015-2026','Obligasi 3Y')

frames = {}
if ob10 is not None: frames['ob10y_yield_pct']=ob10.set_index('year_month')['yield_pct']
if ob3 is not None: frames['ob3y_yield_pct']=ob3.set_index('year_month')['yield_pct']
df_inv = pd.DataFrame(frames).reset_index().rename(columns={'index':'year_month'}).sort_values('year_month')
df_inv.to_csv(PROC/'investment_clean.csv', index=False)
print(f"Investment: {len(df_inv)} baris saved")

### Visualisasi: Return IHSG & Yield Obligasi

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

ax1 = axes[0]
ax1.plot(df_ihsg.Date, df_ihsg.Close, 'b-', lw=1.5)
ax1.fill_between(df_ihsg.Date, df_ihsg.Close, alpha=0.1, color='blue')
ax1.set_title('IHSG Monthly Close Price (2010-2025)', fontweight='bold', fontsize=14)
ax1.set_ylabel('Harga Close')

ax2 = axes[1]
if ob10 is not None:
    ob10_plot = ob10.copy(); ob10_plot['Date'] = pd.to_datetime(ob10_plot.year_month)
    ax2.plot(ob10_plot.Date, ob10_plot.yield_pct, 'r-', lw=1.5, label='Obligasi 10Y')
if ob3 is not None:
    ob3_plot = ob3.copy(); ob3_plot['Date'] = pd.to_datetime(ob3_plot.year_month)
    ax2.plot(ob3_plot.Date, ob3_plot.yield_pct, 'g-', lw=2, label='Obligasi 3Y (dari Mei 2024)')
ax2.set_title('Yield Obligasi Pemerintah (%)', fontweight='bold', fontsize=14)
ax2.set_ylabel('Yield (%)'); ax2.legend()
ax2.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f%%'))

plt.tight_layout(); plt.show()
print(f"\nNote: Data Obligasi 3Y baru tersedia dari Mei 2024.")
print(f"Periode sebelumnya = NaN (TIDAK diisi backfill).")

---
## Ringkasan Output

Semua file processed telah disimpan di `data/processed/`.

In [ ]:
EXPECTED = ['cpi_monthly.csv','cpi_clean.csv','cpi_sektor_monthly.csv','cpi_sektor_multiplier.csv',
            'mortality_clean.csv','ae_ratio_clean.csv','salary_clean.csv','salary_growth.csv',
            'ihsg_monthly.csv','ihsg_annual.csv','investment_clean.csv']
print(f"{'File':<40} {'Size':>10}")
print('-'*52)
for f in EXPECTED:
    p = PROC/f
    if p.exists():
        print(f"  [OK] {f:<36} {p.stat().st_size/1024:6.1f} KB")
    else:
        print(f"  [!!] {f:<36} MISSING")